In [13]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

In [14]:
# ==========================================
# 0. Глобальные параметры
# ==========================================
K_NEIGHBORS_USERS = 50      
K_NEIGHBORS_ITEMS = 20      
TOP_X_RECS = 10             
TEST_SIZE = 0.2
SAMPLE_USER_ID = 2

ALPHA = 0.6                 # Вес User-Based
INTERSECTION_BONUS = 0.3    # Бонус за пересечение UB и IB
SIMILARITY_THRESHOLD = 0.2  # Порог отсечения шума

def load_movielens_data():
    ratings = pd.read_csv('ratings.dat', sep='::', engine='python',
                          names=['userId', 'movieId', 'rating', 'timestamp'], encoding='ISO-8859-1')
    movies = pd.read_csv('movies.dat', sep='::', engine='python',
                         names=['movieId', 'title', 'genres'], encoding='ISO-8859-1')
    users = pd.read_csv('users.dat', sep='::', engine='python',
                        names=['userId', 'gender', 'age', 'occupation', 'zip-code'], encoding='ISO-8859-1')
    return ratings, movies, users

def create_matrices(df):
    # Матрица Пользователь x Фильм (для User-Based)
    pivot_users = df.pivot(index='userId', columns='movieId', values='rating').fillna(0)
    sparse_users = csr_matrix(pivot_users.values)
    
    # Матрица Фильм x Пользователь (для Item-Based)
    pivot_items = df.pivot(index='movieId', columns='userId', values='rating').fillna(0)
    sparse_items = csr_matrix(pivot_items.values)
    
    return pivot_users, sparse_users, pivot_items, sparse_items

In [15]:
# ==========================================
# 1. User-Based предсказания
# ==========================================
def get_ub_predictions(user_id, train_pivot_users, model_knn_users, k_neighbors):
    if user_id not in train_pivot_users.index:
        return {}
    
    user_idx = train_pivot_users.index.get_loc(user_id)
    user_vector = train_pivot_users.iloc[user_idx, :].values.reshape(1, -1)
    
    distances, indices = model_knn_users.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
    
    ub_scores = {}
    user_watched = train_pivot_users.iloc[user_idx, :][train_pivot_users.iloc[user_idx, :] > 0].index.tolist()
    user_avg = train_pivot_users.iloc[user_idx, :][train_pivot_users.iloc[user_idx, :] > 0].mean() if len(user_watched) > 0 else 3.0
    
    for i in range(1, len(distances.flatten())):
        similarity = 1 - distances.flatten()[i]
        if similarity < SIMILARITY_THRESHOLD:
            continue
            
        neighbor_idx = indices.flatten()[i]
        neighbor_ratings_vec = train_pivot_users.iloc[neighbor_idx, :]
        nonzero_ratings = neighbor_ratings_vec[neighbor_ratings_vec > 0]
        
        for movie_id, rating in nonzero_ratings.items():
            if movie_id not in user_watched:
                if movie_id not in ub_scores:
                    ub_scores[movie_id] = {'weighted_sum': 0, 'weight_sum': 0}
                ub_scores[movie_id]['weighted_sum'] += rating * similarity
                ub_scores[movie_id]['weight_sum'] += similarity
                
    final_ub = {}
    for movie_id, scores in ub_scores.items():
        if scores['weight_sum'] > 0:
            raw_pred = scores['weighted_sum'] / scores['weight_sum']
            norm_pred = raw_pred * 0.8 + user_avg * 0.2
            final_ub[movie_id] = max(1.0, min(5.0, norm_pred))
            
    return final_ub

In [16]:
# ==========================================
# 2. Item-Based предсказания
# ==========================================
def get_ib_predictions(user_id, train_pivot_users, train_pivot_items, model_knn_items, k_neighbors):
    if user_id not in train_pivot_users.index:
        return {}
        
    user_ratings = train_pivot_users.loc[user_id]
    watched_movies = user_ratings[user_ratings > 0].index.tolist()
    
    ib_scores = {}
    
    for movie_id in watched_movies:
        if movie_id not in train_pivot_items.index:
            continue
            
        m_idx = train_pivot_items.index.get_loc(movie_id)
        m_vector = train_pivot_items.iloc[m_idx, :].values.reshape(1, -1)
        
        distances, indices = model_knn_items.kneighbors(m_vector, n_neighbors=k_neighbors + 1)
        user_rating = user_ratings[movie_id]
        
        for i in range(1, len(distances.flatten())):
            similarity = 1 - distances.flatten()[i]
            if similarity < SIMILARITY_THRESHOLD:
                continue
                
            neighbor_movie_id = train_pivot_items.index[indices.flatten()[i]]
            
            if neighbor_movie_id not in watched_movies:
                if neighbor_movie_id not in ib_scores:
                    ib_scores[neighbor_movie_id] = {'weighted_sum': 0, 'weight_sum': 0}
                
                ib_scores[neighbor_movie_id]['weighted_sum'] += user_rating * similarity
                ib_scores[neighbor_movie_id]['weight_sum'] += similarity
                
    final_ib = {}
    for movie_id, scores in ib_scores.items():
        if scores['weight_sum'] > 0:
            final_ib[movie_id] = max(1.0, min(5.0, scores['weighted_sum'] / scores['weight_sum']))
            
    return final_ib

In [17]:
# ==========================================
# 3. Гибридное объединение
# ==========================================
def get_hybrid_recommendations(user_id, train_pivot_users, train_pivot_items, 
                               model_knn_users, model_knn_items, movie_titles, top_n=10, alpha=0.6):
    
    ub_preds = get_ub_predictions(user_id, train_pivot_users, model_knn_users, K_NEIGHBORS_USERS)
    ib_preds = get_ib_predictions(user_id, train_pivot_users, train_pivot_items, model_knn_items, K_NEIGHBORS_ITEMS)
    
    all_recommended_movies = set(ub_preds.keys()).union(set(ib_preds.keys()))
    
    hybrid_scores = []
    
    for movie_id in all_recommended_movies:
        score_ub = ub_preds.get(movie_id, 0.0)
        score_ib = ib_preds.get(movie_id, 0.0)
        
        in_both = 1.0 if (score_ub > 0 and score_ib > 0) else 0.0
        
        if score_ub > 0 and score_ib == 0:
            final_score = score_ub
        elif score_ib > 0 and score_ub == 0:
            final_score = score_ib
        else:
            final_score = (alpha * score_ub) + ((1 - alpha) * score_ib) + (INTERSECTION_BONUS * in_both)
            
        final_score = max(1.0, min(5.0, final_score))
        
        hybrid_scores.append({
            'movieId': movie_id,
            'title': movie_titles.get(movie_id, "Unknown"),
            'predicted_rating': final_score,
            'ub_score': score_ub,
            'ib_score': score_ib,
            'in_both': bool(in_both)
        })
        
    hybrid_scores.sort(key=lambda x: x['predicted_rating'], reverse=True)
    return hybrid_scores[:top_n]

In [18]:
# ==========================================
# 4. Оценка качества (Метрики)
# ==========================================
def evaluate_hybrid_model(test_df, train_pivot_users, train_pivot_items, model_knn_users, model_knn_items, movie_titles, k_users, k_items, top_k=10):
    actual_ratings = []
    predicted_ratings = []
    
    precisions = []
    recalls = []
    ndcgs = []
    
    test_users = test_df['userId'].unique()
    sample_users = np.random.choice(test_users, min(200, len(test_users)), replace=False)
    
    for user_id in sample_users:
        user_test_data = test_df[test_df['userId'] == user_id]
        relevant_movies = set(user_test_data[user_test_data['rating'] >= 4.0]['movieId'].tolist())
        
        if not relevant_movies:
            continue
            
        # Вызов с именованными аргументами для надежности
        recs = get_hybrid_recommendations(
            user_id=user_id, 
            train_pivot_users=train_pivot_users, 
            train_pivot_items=train_pivot_items,
            model_knn_users=model_knn_users, 
            model_knn_items=model_knn_items, 
            movie_titles=movie_titles, 
            top_n=top_k
        )
        
        recommended_movies = [r['movieId'] for r in recs]
        
        for r in recs:
            mid = r['movieId']
            actual = user_test_data[user_test_data['movieId'] == mid]['rating']
            if not actual.empty:
                predicted_ratings.append(r['predicted_rating'])
                actual_ratings.append(actual.values[0])
                
        hits = len(set(recommended_movies).intersection(relevant_movies))
        
        precisions.append(hits / top_k if top_k > 0 else 0)
        recalls.append(hits / len(relevant_movies) if len(relevant_movies) > 0 else 0)
        
        dcg = 0.0
        for i, mid in enumerate(recommended_movies):
            if mid in relevant_movies:
                rel = user_test_data[user_test_data['movieId'] == mid]['rating'].values[0]
                dcg += (rel) / np.log2(i + 2)
                
        ideal_rels = sorted(user_test_data[user_test_data['movieId'].isin(relevant_movies)]['rating'].tolist(), reverse=True)[:top_k]
        idcg = sum([(rel) / np.log2(i + 2) for i, rel in enumerate(ideal_rels)])
        
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    rmse = sqrt(mean_squared_error(actual_ratings, predicted_ratings)) if actual_ratings else 0
    
    return {
        'RMSE': rmse,
        'Precision@K': np.mean(precisions),
        'Recall@K': np.mean(recalls),
        'NDCG@K': np.mean(ndcgs)
    }

In [20]:
# ==========================================
# 5. Основная программа
# ==========================================
if __name__ == "__main__":
    print("Загрузка данных...")
    ratings_df, movies_df, users_df = load_movielens_data()
    movie_titles = dict(zip(movies_df['movieId'], movies_df['title']))
    
    print("Разделение на train/test...")
    train_data, test_data = train_test_split(ratings_df, test_size=TEST_SIZE, random_state=42)
    
    print("Создание матриц...")
    train_pivot_users, train_sparse_users, train_pivot_items, train_sparse_items = create_matrices(train_data)
    
    print("Обучение User-Based модели...")
    model_knn_users = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=K_NEIGHBORS_USERS, n_jobs=-1)
    model_knn_users.fit(train_sparse_users)
    
    print("Обучение Item-Based модели...")
    model_knn_items = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=K_NEIGHBORS_ITEMS, n_jobs=-1)
    model_knn_items.fit(train_sparse_items)
    
    print(f"\nГенерация рекомендаций для пользователя ID={SAMPLE_USER_ID}...")
    
    # ЯВНЫЙ ВЫЗОВ С ИМЕНОВАННЫМИ АРГУМЕНТАМИ (гарантирует отсутствие ошибок позиционирования)
    hybrid_recs = get_hybrid_recommendations(
        user_id=SAMPLE_USER_ID, 
        train_pivot_users=train_pivot_users, 
        train_pivot_items=train_pivot_items, 
        model_knn_users=model_knn_users, 
        model_knn_items=model_knn_items, 
        movie_titles=movie_titles, 
        top_n=10
    )
    
    print("\nТоп-10 гибридных рекомендаций:")
    for i, rec in enumerate(hybrid_recs, 1):
        in_both_str = " [UB+IB]" if rec['in_both'] else ""
        print(f"{i:2}. {rec['title']:<45} (Оценка: {rec['predicted_rating']:.2f}, UB: {rec['ub_score']:.2f}, IB: {rec['ib_score']:.2f}){in_both_str}")
        
    print("\n" + "="*60)
    print("ОЦЕНКА КАЧЕСТВА МОДЕЛИ (на подвыборке 200 пользователей)")
    print("="*60)
    
    metrics = evaluate_hybrid_model(
        test_df=test_data, 
        train_pivot_users=train_pivot_users, 
        train_pivot_items=train_pivot_items, 
        model_knn_users=model_knn_users, 
        model_knn_items=model_knn_items, 
        movie_titles=movie_titles, 
        k_users=K_NEIGHBORS_USERS, 
        k_items=K_NEIGHBORS_ITEMS, 
        top_k=10
    )
    
    print(f"RMSE:          {metrics['RMSE']:.4f}")
    print(f"Precision@10:  {metrics['Precision@K']:.4f}")
    print(f"Recall@10:     {metrics['Recall@K']:.4f}")
    print(f"NDCG@10:       {metrics['NDCG@K']:.4f}")

Загрузка данных...
Разделение на train/test...
Создание матриц...
Обучение User-Based модели...
Обучение Item-Based модели...

Генерация рекомендаций для пользователя ID=2...

Топ-10 гибридных рекомендаций:
 1. Lawrence of Arabia (1962)                     (Оценка: 5.00, UB: 4.55, IB: 5.00) [UB+IB]
 2. Christmas Story, A (1983)                     (Оценка: 5.00, UB: 4.75, IB: 5.00) [UB+IB]
 3. Rudy (1993)                                   (Оценка: 4.99, UB: 4.48, IB: 5.00) [UB+IB]
 4. Elizabeth (1998)                              (Оценка: 4.88, UB: 4.30, IB: 5.00) [UB+IB]
 5. Insider, The (1999)                           (Оценка: 4.85, UB: 4.26, IB: 5.00) [UB+IB]
 6. Searching for Bobby Fischer (1993)            (Оценка: 4.85, UB: 4.25, IB: 5.00) [UB+IB]
 7. Schindler's List (1993)                       (Оценка: 4.84, UB: 4.62, IB: 4.44) [UB+IB]
 8. Godfather, The (1972)                         (Оценка: 4.78, UB: 4.61, IB: 4.29) [UB+IB]
 9. Sixth Sense, The (1999)                      